<a href="https://colab.research.google.com/github/gostek198/podstawy-llm-lab-eiti-pw/blob/main/lab01/lab01_yelp_klasyfikacja_LAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#LLM - Klasyczne metody klasyfikacji tekstu - LAB

#Zadanie

Zaadaptuj kod z notatnika *LLM - Klasyczne metody klasyfikacji tekstu - Omówienie* do problemu klasyfikacji liczby gwiazdek dla opinii z serwisu Yelp.
Możesz przygotować pętlę treningową albo w czystym PyTorchu, albo z wykorzystaniem biblioteki PyTorch Lightning.

* Wykorzystaj zbiór `Yelp/yelp_review_full` ([link](https://huggingface.co/datasets/Yelp/yelp_review_full)) zawierający opinie z serwisu Yelp (kolumna: `text`) i etykietę (kolumna: `label`) o wartościach $0,1,2,3,4$ określającą liczbę gwiazdek przyznaną przez użytkownika (a ściślej, liczbę gwiazdek minus jeden). Ponieważ mamy pięć klas, ostatnia warstwa liniowa w sieci neuronowej musi zwracać pięć wartości.
    * Zgodnie z dobrą praktyką z części treningowej wydziel dodatkową część walidacyjną i testową.
    * Ogranicz rozmiar każdej części zbioru danych (treningowej, walidacyjnej i testowej). Część treningowa nie powinna zawierać więcej niż 100k elementów.
* Do ekstrakcji cech z tekstu wykorzystaj **metodę TF-IDF** (*term frequency-inverse document frequency*) opartą o podejście typu worek słów (*bag-of-words*). Zastosuj funkcję `TfidfVectorizer` z biblioteki `scikit-learn`.
* Wykorzystaj podobną architekturę sieci (perceptron wielowarstwowy z warstwą Dropout) jak w notatniku wykładowym.



## Punkty do wykonania

1.   Napisz funkcję znajdującą i wyświetlającą $k$ elementów zbioru testowego dla których model najbardziej się myli, czyli estymuje najmniejsze prawdopodobieństwa prawdziwej klasy. Softmax jest funkcją ściśle rosnącą, więc wystarczy znaleźć elementy z najmniejszą wartością nieznormalizowanego wyjścia z sieci (logita) dla prawdziwej klasy.
2.   Zbadaj wpływ wybranych parametrów funkcji ekstrakcji cech z tekstu `TfidfVectorizer` na skuteczność wytrenowanego modelu. Uruchom kilka eksperymentów z różnymi wartościami parametrów i porównaj dokładność wytrenowanego modelu na zbiorze walidacyjnym.
3.   Zbadaj wpływ wybranych hiperparametrów modelu (np. liczba warstw liniowych modelu, rozmiary warstw) i procesu uczenia (np. początkowa wartość stopy uczenia, liczba epok, typ i parametry planisty stopy uczenia, typ i parametry optymalizatora) na skuteczność wytrenowanego modelu. Uruchom kilka eksperymentów z różnymi wartościami hiperparametrów i porównaj dokładność wytrenowanego modelu na zbiorze walidacyjnym. Następnie wykonaj finalną ewaluację najlepszego modelu na zbiorze testowym.


##Przygotowanie środowiska
Upewnij się, że notatnik jest uruchomiony na maszynie z GPU. Jesli GPU nie jest dostępne zmień typ maszyny (Runtime | Change runtime type) i wybierz T4 GPU.

In [1]:
!nvidia-smi

Tue Nov 10 10:24:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Instalacja dodatkowych bibliotek: datasets (z biblioteki HuggingFace), TorchMetrics i W&B (Weights and Biases) Models.

In [2]:
!pip install -q datasets
!pip install -q torchmetrics
!pip install -q wandb

Import bibliotek.

In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"Wersja biblioteki PyTorch: {torch.__version__}")

Wersja biblioteki PyTorch: 2.14.1+cu130


Sprawdzenie dostępności GPU.

In [4]:
print(f"Dostępność GPU: {torch.cuda.is_available()}")
print(f"Typ GPU: {torch.cuda.get_device_name(0)}")

Dostępność GPU: True
Typ GPU: Tesla T4


In [5]:
import wandb

# Logowanie do serwisu Weights&Biases monitorującego przebieg eksperymentów
wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: gostek198 (gostek198-warsaw-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Rozwiązanie

# 1.Dane

In [7]:
#Wczytanie danych na ktorych model ma zaczac sie uczyc - pokazanie danych - (podczas trenignu model) tysiące opinii z
#prawidłową liczbą gwiazdek, a on sam wyłapuje zależności, np. że słowa „terrible” albo „worst” zwykle oznaczają 1 gwiazdkę, a „amazing” 5.

from datasets import load_dataset

SEED = 42  # stałe ziarno - każdy przebieg da ten sam podział danych
dataset = load_dataset("Yelp/yelp_review_full")
print(dataset)
print(dataset["train"].features)

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})
{'label': ClassLabel(names=['1 star', '2 star', '3 stars', '4 stars', '5 stars']), 'text': Value('string')}


#2. Podzial danych na zbior treningowy,walidacyjny i testowy

In [8]:
#Z 650 tysięcy biore losowo 70 tysięcy i dziele je na trzy rozłączne części.

# - train (50k) - na nich model się uczy
# - val   (10k) - sprawdzamy model w trakcie nauki
# - test  (10k) - końcowy "egzamin", używany tylko raz, na samym końcu

# Zgodnie z treścią zadania val i test z części treningowej.
# 50k zamiast limitu 100k - macierz cech TF-IDF zamienimy później na gęstą:
# 50 000 x 10 000 słów x 4 bajty = ok. 2 GB RAM (przy 100k byłoby 4 GB i Colab mógłby się wysypać)

import numpy as np

N_TRAIN = 50_000
N_VAL = 10_000
N_TEST = 10_000

# 1) Z części treningowej usuwam 20k próbek na walidację + test.
#    stratify_by_column="label" -> w każdej części zachowane są te same proporcje klas (gwiazdek)
split = dataset["train"].train_test_split(
    test_size=N_VAL + N_TEST, seed=SEED, stratify_by_column="label"
)

# 2) Odcięte 20k dziele po równo: 10k walidacja, 10k test
val_test = split["test"].train_test_split(
    test_size=N_TEST, seed=SEED, stratify_by_column="label"
)

# 3) Z pozostałej części treningowej losuje 50k próbek
train_dataset = split["train"].shuffle(seed=SEED).select(range(N_TRAIN))
val_dataset = val_test["train"]
test_dataset = val_test["test"]

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Train: 50000, Val: 10000, Test: 10000


In [9]:
# sprawdzam czy ilosc poszczegolnych gwiazdek jest podobna miedzy danymi recenzjami
#czy klasy sa rownowazne ?

class_names = dataset["train"].features["label"].names
print(class_names)

for name, ds in [("train", train_dataset), ("val", val_dataset), ("test", test_dataset)]:
    counts = np.bincount(ds["label"], minlength=5)  # zliczenie wystąpień etykiet 0..4
    print(f"{name:5s}: {counts}")

# Podgląd przykładowej opinii
sample = train_dataset[0]
print(f"\nPrzykład:\nlabel = {sample['label']} ({class_names[sample['label']]})\ntext = {sample['text'][:300]}")


#ten sam rozklad klas w czesci val i test -> stratyfikacja, aby wyniki mogly byc porownywalne

['1 star', '2 star', '3 stars', '4 stars', '5 stars']
train: [10099  9906  9965  9918 10112]
val  : [2000 2000 2000 2000 2000]
test : [2000 2000 2000 2000 2000]

Przykład:
label = 3 (4 stars)
text = Nice take on creole. Not quite as good as the real thing at the Big Easy, but last time I checked you have to fly across the country for that sort of thing. \nI didn't realize this was fine dining when I walked in, so heads up, wear something besides your ripped blue jeans. It's an odd location for 


# 3. Ekstrakcaj cech metodą TF-IDF


In [12]:
# TF-IDF (term frequency-inverse document frequency) to metoda pozwalajaca na ekstrakcje cech z tekstu. Wykorzystuje podejscie typu worke slow (bag of words).
# Ta metoda wyznacza wage istotnosci slowa w dokumencie ktory jest czescia wiekszego zbioru dokuemntow

# TF-IDF zamienia każdą opinię na wektor liczb o długości vocab_size: (siec neuronowa operuje na liczbach)
# każda kolumna odpowiada jednemu słowu ze słownika,
# wartość = waga słowa w tej opinii:

# Słowa występujące wszędzie (np. "the", "and") dostają małą wagę - niewiele mówią o ocenie.
# To podejście "worek słów" (bag-of-words) - kolejność słów jest pomijana.

#Funkcja TfidfVectorizer z biblioteki scikit-learn tworzy rzadką macierz cech TF-IDF na podstawie zbioru dokumentów testowych.
# Każdy wiersz macierzy reprezentuje jeden dokument ze zbioru a kolumny zawierają wagi każdego słowa wyznaczone metodą TF-IDF.

from sklearn.feature_extraction.text import TfidfVectorizer

vocab_size = 10_000

vectorizer = TfidfVectorizer(
    max_features=vocab_size,   # słownik = 10k najczęstszych słów
    lowercase=True,            # "Great" i "great" to to samo słowo
    analyzer="word",           # analiza na poziomie słów, nie znaków
    ngram_range=(1, 1),        # tylko pojedyncze słowa (unigramy)
    stop_words="english",      # bez (a, the, and...)
)

# fit_transform TYLKO dla train: budowa słownik i liczenie IDF.
# dla danych val i test dane nie moga byc podgladane aby zapewnic poprawna jakosc predykcji modelu

train_tfidf = vectorizer.fit_transform(train_dataset["text"])
val_tfidf = vectorizer.transform(val_dataset["text"])
test_tfidf = vectorizer.transform(test_dataset["text"])

print(f"Train: {train_tfidf.shape}, Val: {val_tfidf.shape}, Test: {test_tfidf.shape}")

Train: (50000, 10000), Val: (10000, 10000), Test: (10000, 10000)


In [11]:
# Jakie słowa trafiły do słownika i jak rzadka jest macierz cech.

feature_names = vectorizer.get_feature_names_out()
print(f"Przykładowe słowa ze słownika: {np.random.choice(feature_names, 15)}\n")

# Liczba niezerowych cech w każdej opinii - na podstawie notatnika wykladowego
non_zero_counts = np.diff(train_tfidf.indptr)
print(f"Średnia liczba niezerowych cech na opinię: {non_zero_counts.mean():.1f} z {vocab_size}")

# Cechy przykładowej opinii - 10 słów o największej wadze
ndx = 0
row = train_tfidf[ndx]
top = row.toarray()[0].argsort()[::-1][:10]
print(f"\nOpinia {ndx} ({class_names[train_dataset[ndx]['label']]}): najważniejsze słowa wg TF-IDF:")
for i in top:
    print(f"  {feature_names[i]:15s} {row[0, i]:.3f}")

Przykładowe słowa ze słownika: ['mein' 'minced' 'nick' 'incorrectly' 'pizzeria' 'mo' '250' 'drivers'
 'ask' 'sliding' 'criticism' 'ing' 'ngetting' 'crumbly' 'wildly']

Średnia liczba niezerowych cech na opinię: 47.9 z 10000

Opinia 0 (4 stars): najważniejsze słowa wg TF-IDF:
  creole          0.317
  jeans           0.258
  thing           0.255
  ripped          0.252
  fly             0.249
  heads           0.248
  wear            0.229
  country         0.220
  realize         0.211
  odd             0.209
